# Bayesian Networks

Consider the following (very simplified) model of molecular biology. A biological function $F$ depends, at the molecular level, on four types of molecule. At the lowest level are the *genes*, $G$, which are the basic "code of life". The genes are transcribed (read) to produce *RNA* ($R$) which has two roles. Non-coding RNA can perform functions by itself, whilst messenger RNA (mRNA) enables the production of *proteins* $P$. Proteins also perform functions, and are modified by small molecules called *ligands* $L$.

Biology is noisy and messy with many factors that can affect, for example, the transcription of a gene into the RNA that it codes for, and so it makes good sense to model the five things $(G,R,P,L,F)$ as random variables with a corresponding joint probability distribution $P(G,R,P,L,F)$.

This distribution would be formidably hard to work with. Principally, it would be incredibly difficult to determine what the values in the distribution are. In humans there are roughly:

* Tens of thousands of genes
* Hundreds of thousands of RNA strands
* Tens of thousands of proteins
* Hundreds of thousands of ligands
* Hundreds of thousands of functions

The total number of elements in the multidimensional array (often called a tensor) $P(G,R,P,L,F)$ is the product of the numbers of possible values of each variable, which is of order $10^{4}\times10^{5}\times10^{4}\times10^{5}\times10^{5}=10^{23}$ - intractable. Even at 8 bytes per element, simply storing this would require of the order of $10^{24}$ bytes (a trillion terabytes).

In [1]:
import numpy as np

# Orders of magnitude for the number of possible values of G, R, P, L, F
sizes = {'G': 1e4, 'R': 1e5, 'P': 1e4, 'L': 1e5, 'F': 1e5}

full = np.prod(list(sizes.values()))
print(f"Elements in the full joint distribution: {full:.0e}")
print(f"Storage at 8 bytes per element:          {8*full:.0e} bytes = {8*full/1e12:.0e} terabytes")

# If all five variables were independent we would store one distribution per variable
independent = sum(sizes.values())
print(f"Parameters if independent:               {independent:.1e}   ({8*independent/1e6:.1f} MB)")

Elements in the full joint distribution: 1e+23
Storage at 8 bytes per element:          8e+23 bytes = 8e+11 terabytes
Parameters if independent:               3.2e+05   (2.6 MB)


Although this might seem like a hopeless situation, we do have some useful tools to help us overcome this. For example, if we were to assume statistical independence $P(G,R,P,L,F) = P(G)P(R)P(P)P(L)P(F)$ then the number of parameters in each of these distributions is the number of possibilities for that "thing" (eg $P(G)$ will have tens of thousands of elements). The total number of parameters would be reduced to a few hundred thousand, requiring only a few megabytes of storage, which is trivial for us today.

The assumption of independence that we would have to make to do this would, however, not be reasonable or acceptable. We **know** that there are clear links between some of these things and they are definitely not independent: the probability that a particular protein is expressed definitely depends on what RNA sequences are present, and these in turn depend on what genes are present.

Fortunately, there are options open to us beyond treating the random variables as independent. In fact, the product rule of probability allows us a large amount of freedom over how we model the distribution. Recalling the definition of the product rule $P(X,Y)=P(X\vert Y)P(Y)$, and applying it repeatedly (the chain rule), we can factorise $P(G,R,P,L,F)$ *exactly* as a product of conditional distributions for *any* ordering of the variables, for example

$$
P(G,R,P,L,F) = P(L\vert R,P,G,F)P(R\vert P,G,F)P(P\vert G,F)P(G\vert F)P(F)
$$

On its own this does not help: the last factor of a full chain rule has as many parameters as the joint distribution itself. The savings come when we are able to *drop* some of the variables from the conditioning, because the variable is (conditionally) independent of them. For example, the following are all possible modelling choices (and there are many more possibilities):

* $P(G,R,P,L,F) = P(L\vert R)P(R\vert P)P(P\vert G)P(G\vert F)P(F)$
* $P(G,R,P,L,F) = P(G\vert L)P(L\vert F)P(F\vert P)P(P\vert R)P(R)$
* $P(G,R,P,L,F) = P(R\vert L)P(L\vert G)P(G\vert F)P(F\vert P)P(P)$

Each of these is an accurate representation of the joint distribution *only if* the conditional independence assumptions that it makes are true, and so some of them will be much better models than others. We can see this on a small example. The chain rule always reproduces a joint distribution exactly, whereas a factorisation in which we drop a variable from the conditioning reproduces it only if the independence assumption is true. In the notation we use in code, $P(A,B\vert C)$ is `PAB_C` and arrays are indexed in the same order as the name.

In [2]:
rng = np.random.default_rng(1)

# A random joint distribution of three binary variables, PXYZ[x][y][z]
PXYZ = rng.random((2, 2, 2))
PXYZ /= PXYZ.sum()

# Chain rule: P(X,Y,Z) = P(X) P(Y|X) P(Z|X,Y), with no assumptions
PXY = PXYZ.sum(axis=2)
PX = PXY.sum(axis=1)
PY_X = (PXY / PX[:, None]).T                  # PY_X[y][x]
PZ_XY = (PXYZ / PXY[:, :, None]).transpose(2, 0, 1)   # P(Z | X,Y)[z][x][y]
PXYZ_chain = np.einsum('x,yx,zxy->xyz', PX, PY_X, PZ_XY)
print("Chain rule reproduces the joint exactly:", np.allclose(PXYZ, PXYZ_chain))

# Dropping X from the conditioning of Z, i.e. assuming P(Z | X,Y) = P(Z | Y)
PYZ = PXYZ.sum(axis=0)                        # P(Y,Z)[y][z]
PY = PYZ.sum(axis=1)
PZ_Y = (PYZ / PY[:, None]).T                  # PZ_Y[z][y]
PXYZ_dropped = np.einsum('x,yx,zy->xyz', PX, PY_X, PZ_Y)
print("Dropping a variable reproduces the joint:", np.allclose(PXYZ, PXYZ_dropped), " (not in general)")

Chain rule reproduces the joint exactly: True
Dropping a variable reproduces the joint: False  (not in general)


Why is this useful and desirable? There are two pragmatic reasons for this:

1. The number of parameters needed to characterise the distribution is much reduced.

2. It is rarely possible, in real problems, to measure the joint distribution directly. In most practical situations we may observe a small number of variables whilst controlling the others. Factorisation of the joint distribution in terms of things that we can observe may, in some circumstances, be important if not necessarily optimal.

However, neither of these reasons helps us to answer an important question: **which factorisation should we choose**? To answer this question, we have to think hard about the problem we are working with, and about the relationships between the variables. What do we know?

First, we know that the presence of RNA results from transcription of the genes. It doesn't depend on any of the other variables. It therefore seems plausible that our factorisation should include $P(R\vert G)$.

Secondly, in our model the genes have no causes among the other variables, so including the unconditional distribution $P(G)$ seems reasonable.

Thirdly, proteins depend on RNA, so we might include $P(P\vert R)$. But they are also dependent on the ligands (which modify them) so the precise protein present depends on both $R$ and $L$, so we should include $P(P\vert R,L)$.

Ligands themselves have no causes among the other variables (they are just there) and so we include the unconditional distribution $P(L)$.

Finally, function $F$ depends directly on both proteins and RNA, so we include $P(F\vert P,R)$.

This gives us a factorisation:

$P(G,R,P,L,F) = P(F\vert P,R)P(P\vert R,L)P(R\vert G)P(L)P(G)$

In graphical form, this can be drawn as

![Example of a DAG](./figures/SimpleBayesian.png)

This is an example of a **Bayesian Network**. It is a valid factorisation of the joint distribution, and it is structured in such a way as to reflect the **causal relationships** in the problem.

## Formal definition of a Bayesian Network

A Bayesian network is a representation of a suitably factorised probability distribution as a *directed graph* in which

1. Each node represents a single random variable (discrete or continuous).
2. Edges between nodes represent conditional relationships and are directed from the "conditioner" (or *parent*) node to the "conditioned" node.
3. Each node has an associated conditional probability distribution, with the variable associated with the node conditioned on its parents.
4. "Orphan" (or *root*) nodes with no parents have unconditional (prior) distributions: they are not conditioned on any other variable. (A root node is independent of its non-descendants, but of course not of its descendants.)
5. The graph must be strictly acyclic: it is a **directed acyclic graph** (DAG).

The DAG naturally and visually conveys the structure and relationships within the factorised joint distribution, and it is trivial to map between the DAG and the factorised distribution. The edges in the graph convey "influence" or, more strongly, "causation" (genes cause RNA, RNA and proteins cause function). Since nodes in the graph represent variables, each node has an associated probability distribution that is conditioned on the incoming arrows.

One of the nicest features of Bayesian Networks is that there is a unique, unambiguous relationship between the diagram and the factorisation of the joint probability distribution. This means that all of your "thinking" can be done using the diagrams, and when you are satisfied that you have a diagram that represents the relationships, you can simply write down the factorisation. (Note that the mapping is unique in this direction only: different DAGs, for example those arising from different orderings, can represent the same distribution, and a DAG is a good model only if the independence assumptions that it encodes are actually true of the problem.)

The following function uses `pyvis` to draw a DAG from a dictionary of node positions and a list of edges (parent, child). The result is interactive: nodes can be dragged with the mouse. We will use it to draw the networks in the rest of this section and in the exercises. (If you do not have `pyvis`, install it with `pip install pyvis`.)

In [3]:
import html as _html
from pyvis.network import Network      # pip install pyvis
from IPython.display import HTML, display

def draw_dag(pos, edges, title=None, scale=110, cdn_resources='remote'):
    """Draw a DAG interactively with pyvis (the nodes can be dragged with the mouse).
    pos: {node: (x, y)} with y increasing upwards, edges: list of (parent, child).
    cdn_resources='remote' loads the vis.js library from the internet; use 'in_line' to embed it (offline use)."""
    xs, ys = zip(*pos.values())
    height = int((max(ys) - min(ys)) * scale + 140)
    net = Network(height=f"{height}px", width="100%", directed=True, notebook=True,
                  cdn_resources=cdn_resources)
    for n, (x, y) in pos.items():
        net.add_node(n, label=n, x=x*scale, y=-y*scale, physics=False, shape="circle",
                     color={"background": "#E6F1FB", "border": "#378ADD"},
                     font={"size": 20, "color": "#042C53"}, margin=12)
    for parent, child in edges:
        net.add_edge(parent, child, color="#555555", width=2, arrows="to")
    net.set_options('{"physics": {"enabled": false}, "edges": {"smooth": false}}')
    page = net.generate_html(notebook=True)
    heading = f"<p><b>{title}</b></p>" if title else ""
    display(HTML(f'<div>{heading}<iframe srcdoc="{_html.escape(page, quote=True)}" width="100%" '
                 f'height="{height + 40}" style="border:0"></iframe></div>'))

# The molecular biology network: P(G,R,P,L,F) = P(F|P,R) P(P|R,L) P(R|G) P(L) P(G)
draw_dag({'G': (0, 2), 'R': (0, 1), 'L': (2, 1), 'P': (1, 0), 'F': (0, -1)},
         [('G', 'R'), ('R', 'P'), ('L', 'P'), ('P', 'F'), ('R', 'F')], title='Molecular biology network')

## A method for constructing Bayesian Networks

There is a nice well-defined recipe for constructing a Bayesian network which amounts to applying the chain rule, but choosing for each variable only those predecessors that directly influence it.

Let us begin by reminding ourselves of the conditional factorisation of the joint distribution. We assume an arbitrary ordering of the variables which can be changed at will.

$P(X_1,\dots,X_N) = P(X_1)\prod_{i=2}^N P(X_i\vert X_{i-1},\dots,X_1)$.

This is called the **chain rule**. It requires an ordering to be imposed on the random variables and we will write as a shorthand that

$P(X_i\vert X_{i-1},\dots,X_1) = P(X_i\vert\mathrm{parents}(X_i))$

where $\mathrm{parents}(X_i)$ is a subset of $\{X_{i-1},\dots,X_1\}$. We can do this very simply in the graph by making sure that the nodes are ordered so that every node comes after all of its parents (a *topological ordering*): nodes with no incoming edges (orphans) come first, and a node with incoming edges comes later than the nodes that point to it. The question is, how to achieve this?

The chain rule implies that a Bayesian network properly represents its domain if each node is conditionally independent of its predecessors in the ordering, given its parents. Here is a procedure for achieving this:

1. Choose the variables needed to model the domain and order them. A good ordering has causes preceding effects.

2. For each node, choose a set of parents such that the chain rule is satisfied and that the parents can plausibly represent causes of the node.

3. Add edges directed from parents to the node.

4. Write down a conditional probability table for the node.

Let's work through a simple example:

A baker is making bread. The recipe uses an active "starter", which has high activity in more humid conditions. The goal is to get a nice blistered crust on the bread, which depends on the bread being well-risen and also on the humidity. Whether the bread rises well depends on the activity of the starter and the temperature in the kitchen, which also depends on the humidity. The bakery also owns a cat, who likes to sit on the worktop but is strongly discouraged from doing so. We summarise this as follows:

* Humidity (H): Low / High
* StarterActivity (A): Sluggish / Vigorous. Depends on Humidity.
* KitchenTemp (T): Cool / Warm. Depends on Humidity.
* Rise (R): Poor / Good. Depends on StarterActivity and KitchenTemp.
* CrustBlisters (B): No / Yes. Depends on Rise and Humidity.
* CatOnCounter (C): No / Yes. Independent of everything (the bakery cat's whims).

We have six variables here, each of which is a binary random variable. The joint distribution therefore has $2^6-1=63$ independent parameters. Our factorisation will greatly reduce this.

What is the proper ordering of the variables here? We need to work out what the causative chain of events might be.

* Humidity $H$ does not depend on anything else.  
* Kitchen temperature $T$ depends on humidity $H$.  
* Starter activity $A$ depends on humidity $H$.  
* Rise $R$ depends on temperature $T$ and starter activity $A$.  
* CrustBlisters $B$ depends on rise $R$ and humidity $H$.  
* The cat's whereabouts $C$ do not depend on anything.

The factorisation is therefore:

$P(H,A,T,R,B,C) = P(H)P(T\vert H)P(A\vert H)P(R\vert A,T)P(B\vert R,H)P(C)$

![DAG for Baking Bread](./figures/DAG_Baking.png)

Can we ascribe some rough probabilities to these events?

Let's say it is slightly less likely to be high humidity than low humidity,

| $P(H=High)$ | $P(H=Low)$ |
|:------:|:------------:|
| 0.45    | 0.55          |

The cat likes to get on the worktop sometimes, but not most of the time:

| $P(C)$ | $P(\lnot C)$ |
|:------:|:------------:|
| 0.3    | 0.7          |

The starter is more likely to be vigorous when it is humid:

|           | $P(A=Vigorous\vert H)$ | $P(A=Sluggish\vert H)$ |
|----------:|:-------------:|:-------------------:|
| $H=Low$   | 0.4           |  0.6                |
| $H=High$  | 0.75          |  0.25               |

The temperature in the kitchen is more likely to be high when the humidity is high:

|           | $P(T=Warm\vert H)$ | $P(T=Cool\vert H)$ |
|----------:|:-------------:|:-------------------:|
| $H=Low$   | 0.3           |  0.7                |
| $H=High$  | 0.65          |  0.35               |

A good rise is more likely with high temperature and activity:

|                  | $P(R=Good\vert A,T)$ | $P(R=Poor\vert A,T)$ |
|-----------------:|:---------------:|:---------------------:|
| $A=Sluggish, T=Cool $ | 0.05            |  0.95                 |
| $A=Sluggish, T=Warm $ | 0.40            |  0.60                 |
| $A=Vigorous, T=Cool $ | 0.50            |  0.50                 |
| $A=Vigorous, T=Warm $ | 0.92            |  0.08                 |

Finally, a good blistered crust is more likely when the humidity is high and the rise is good:

|                  | $P(B=Yes\vert R,H)$ | $P(B=No\vert R,H)$ |
|-----------------:|:---------------:|:---------------------:|
| $R=Poor, H=Low $ | 0.02            |  0.98                 |
| $R=Poor, H=High $ | 0.10            |  0.90                 |
| $R=Good, H=Low $ | 0.15            |  0.85                 |
| $R=Good, H=High $ | 0.50            |  0.50                 |

There are 14 independent parameters here - far fewer than are needed to describe the full joint distribution.

### The network in code

We store each conditional probability table as a numpy array, using the naming convention of the earlier sections: $P(A\vert H)$ is `PA_H`, $P(R\vert A,T)$ is `PR_AT`, and the arrays are indexed in the same order as the name, so that `PR_AT[r][a][t]` is $P(R=r\vert A=a,T=t)$. The index of a value increases with the value: for $H$, 0 is Low and 1 is High; for $A$, 0 is Sluggish and 1 is Vigorous; for $T$, 0 is Cool and 1 is Warm; for $R$, 0 is Poor and 1 is Good; for $B$ and $C$, 0 is No and 1 is Yes.

In [ ]:
PH = np.array([0.55, 0.45])                      # P(H):   Low, High
PC = np.array([0.7, 0.3])                        # P(C):   No, Yes
PA_H = np.array([[0.6, 0.25],                    # P(A | H): rows A = Sluggish, Vigorous; columns H = Low, High
                 [0.4, 0.75]])
PT_H = np.array([[0.7, 0.35],                    # P(T | H): rows T = Cool, Warm
                 [0.3, 0.65]])
PR_AT = np.array([[[0.95, 0.60], [0.50, 0.08]],  # P(R | A,T)[r][a][t]: R = Poor
                  [[0.05, 0.40], [0.50, 0.92]]]) #                      R = Good
PB_RH = np.array([[[0.98, 0.90], [0.85, 0.50]],  # P(B | R,H)[b][r][h]: B = No
                  [[0.02, 0.10], [0.15, 0.50]]]) #                      B = Yes

# Every conditional distribution must sum to 1 over the value of the variable itself (axis 0)
for name, P in [('PH', PH), ('PC', PC), ('PA_H', PA_H), ('PT_H', PT_H), ('PR_AT', PR_AT), ('PB_RH', PB_RH)]:
    print(f"{name:6s} normalised: {np.allclose(P.sum(axis=0), 1)}")

# Number of free parameters: (number of values - 1) for each combination of parent values
n_params = 14
print("Free parameters in the network:", n_params, "  in the full joint distribution:", 2**6 - 1)

PH     normalised: True
PC     normalised: True
PA_H   normalised: True
PT_H   normalised: True
PR_AT  normalised: True
PB_RH  normalised: True
Free parameters in the network: 14   in the full joint distribution: 63


The joint distribution is the product of the factors, $P(H,A,T,R,B,C)=P(H)P(T\vert H)P(A\vert H)P(R\vert A,T)P(B\vert R,H)P(C)$. We can multiply them using `einsum`, which takes a string saying which axis of each factor belongs to which variable. Note that this builds the whole joint distribution, which is only feasible because this network is small. In the next section we will see how to avoid doing it.

In [5]:
# Joint distribution PHCATRB[h][c][a][t][r][b]
names = ['H', 'C', 'A', 'T', 'R', 'B']
PHCATRB = np.einsum('h,c,ah,th,rat,brh->hcatrb', PH, PC, PA_H, PT_H, PR_AT, PB_RH)
print("Joint distribution has shape", PHCATRB.shape, "and sums to", round(PHCATRB.sum(), 10))

def posterior(PJ, names, query, evidence={}):
    """P(query | evidence) from a joint distribution PJ with one axis per variable in `names`.
    evidence is a dictionary {variable name: value}."""
    index = tuple(evidence.get(n, slice(None)) for n in names)   # fix the evidence variables
    sub = PJ[index]
    remaining = [n for n in names if n not in evidence]
    other_axes = tuple(i for i, n in enumerate(remaining) if n != query)
    p = sub.sum(axis=other_axes)                                 # marginalise the other variables
    return p / p.sum()                                           # normalise

print("P(B)                =", np.round(posterior(PHCATRB, names, 'B'), 4))
print("P(H)                =", np.round(posterior(PHCATRB, names, 'H'), 4))
print("P(H | B=Yes)        =", np.round(posterior(PHCATRB, names, 'H', {'B': 1}), 4))
print("P(A | B=Yes)        =", np.round(posterior(PHCATRB, names, 'A', {'B': 1}), 4))

Joint distribution has shape (2, 2, 2, 2, 2, 2) and sums to 1.0
P(B)                = [0.8026 0.1974]
P(H)                = [0.55 0.45]
P(H | B=Yes)        = [0.1801 0.8199]
P(A | B=Yes)        = [0.1874 0.8126]


Observing a blistered crust raises the probability that the humidity is high from 45% to 82%, and that the starter is vigorous from 56% to 81%. Evidence about an effect flows back "up" the network to its causes.

## What the graph says about independence

The chain rule construction implies that each node is conditionally independent of its other predecessors given its parents. More generally, the three structures from the notes on conditional independence (chain, fork and collider) can be found in any network, and they tell us which variables are independent without doing any calculation.

In the baking network:

* $A\leftarrow H\rightarrow T$ is a fork, so $A$ and $T$ are dependent (both depend on the humidity) but independent given $H$.
* $A\rightarrow R\leftarrow T$ is a collider, so observing the rise $R$ makes $A$ and $T$ dependent even given $H$ (explaining away: if the bread rose badly and the kitchen was warm, the starter was probably sluggish).
* $C$ has no edges, so the cat is independent of everything.

We can confirm these numerically.

In [6]:
def cond_independent(PXYZ, tol=1e-9):
    """Test whether X and Y are conditionally independent given Z, for a joint distribution PXYZ[x][y][z]."""
    PZ = PXYZ.sum(axis=(0, 1))
    PXY_Z = PXYZ / PZ[None, None, :]
    PX_Z = PXY_Z.sum(axis=1)
    PY_Z = PXY_Z.sum(axis=0)
    return np.allclose(PXY_Z, np.einsum('xz,yz->xyz', PX_Z, PY_Z), atol=tol)

# Sum out B and C. Axes of PHATR are now (h, a, t, r)
PHATR = PHCATRB.sum(axis=(1, 5))

# A and T given H (a fork): rearrange to PATH[a][t][h]
PATH = np.einsum('hatr->ath', PHATR)
print("A and T independent given H?    ", cond_independent(PATH))

# A and T marginally
PAT = PHATR.sum(axis=(0, 3))
print("A and T independent?            ", np.allclose(PAT, np.outer(PAT.sum(axis=1), PAT.sum(axis=0))))

# A and T given R and H (R is a collider): combine (r,h) into a single conditioning variable
PATRH = np.einsum('hatr->atrh', PHATR).reshape(2, 2, 4)
print("A and T independent given R, H? ", cond_independent(PATRH))

# C is independent of everything, for example of H
PHC = PHCATRB.sum(axis=(2, 3, 4, 5))
print("C independent of H?             ", np.allclose(PHC, np.outer(PHC.sum(axis=1), PHC.sum(axis=0))))

A and T independent given H?     True
A and T independent?             False
A and T independent given R, H?  False
C independent of H?              True


The same holds for the other nodes. A Bayesian network encodes a set of conditional independence relations, and these are exactly what allows us to reduce the number of parameters from 63 to 14. The graph also tells us what we may *not* assume: dropping the edge $T\rightarrow R$, for example, would assert that the rise does not depend on the temperature given the starter activity, which is false for this problem.

Finding the posterior probability of any variable given evidence, as we have just done by building the full joint distribution, is called *inference*. It is the subject of the next section.